In [1]:
# Install dependencies
!pip install PyMuPDF faiss-cpu sentence-transformers transformers fastapi uvicorn nest-asyncio -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/

In [3]:
# Extraction from PDF file
# As suggested picking 2 chapter from PDF

import fitz

# /content/ConceptsofBiology-WEB.pdf

def extract_chapter(pdf_path, start_page, end_page):
    """Extracts text from specified pages."""
    doc = fitz.open(pdf_path)
    text = ""
    for page_num in range(start_page - 1, end_page):  # 0-indexed
        text += doc[page_num].get_text()
    return text

chapter1 = extract_chapter("/content/ConceptsofBiology-WEB.pdf", 5, 26)
chapter2 = extract_chapter("/content/ConceptsofBiology-WEB.pdf", 27, 54)

In [5]:
len(chapter1)

41641

In [6]:
len(chapter2)

77776

In [9]:
# Chunk the text and generate embeddings

from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import faiss
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
def chunk_text(text, max_tokens=250):
    #Split text into smaller chunks.
    sentences = text.split('. ')
    chunks, chunk = [], ""
    for sentence in sentences:
        if len(chunk) + len(sentence.split()) <= max_tokens:
            chunk += sentence + '. '
        else:
            chunks.append(chunk.strip())
            chunk = sentence + '. '
    if chunk:
        chunks.append(chunk.strip())
    return chunks

docs = chunk_text(chapter1) + chunk_text(chapter2)
embeddings = model.encode(docs)

In [18]:
print(len(docs))
print(len(docs[0]),len(docs[1]))

309
381 280


In [19]:
embeddings

array([[ 0.02655478, -0.04778226,  0.06508278, ..., -0.01167444,
        -0.003441  ,  0.02379477],
       [ 0.05623985, -0.05394854,  0.06358684, ..., -0.0271262 ,
         0.00987955,  0.02464428],
       [ 0.0440748 , -0.02083004,  0.02360018, ..., -0.02998308,
         0.01978103, -0.00322992],
       ...,
       [-0.06912646,  0.11138371, -0.07096279, ...,  0.06542759,
         0.11196699, -0.02219771],
       [-0.01567801, -0.03992388, -0.08173319, ..., -0.04158105,
         0.15093444, -0.01586436],
       [-0.01161968, -0.06137133, -0.06373123, ..., -0.11157116,
         0.19340663, -0.03941122]], dtype=float32)

In [20]:
# Build Index using Faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

In [22]:
# Query Processing with Contextual Retrieval

def search_similar(query, top_k=3):
    #Retrieve top-k similar chunks.#
    query_embedding = model.encode([query])
    distances, indices = index.search(query_embedding, top_k)
    return [docs[i] for i in indices[0]]

In [23]:
query = "What are the properties of life?"
context_docs = search_similar(query)
print("\n\n".join(context_docs))

To make the whole-Earth
image, NASA scientists combine observations of different parts of the planet. (credit: modification of work by NASA)
CHAPTER OUTLINE
they do not meet the criteria that biologists use to define life.
From its earliest beginnings, biology has wrestled with four questions: What are the shared
properties that make something “alive”? How do those various living things function? When faced
with the remarkable diversity of life, how do we organize the different kinds of organisms so that
we can better understand them? And, finally—what biologists ultimately seek to understand—how
did this diversity arise and how is it continuing? As new organisms are discovered every day,
biologists continue to seek answers to these and other questions.
Properties of Life
All groups of living organisms share several key characteristics or functions: order, sensitivity or
response to stimuli, reproduction, adaptation, growth and development, regulation/homeostasis,
energy processing, an

In [24]:
# Using Huggingface LLM

from transformers import pipeline

qa_pipeline = pipeline("text-generation", model="gpt2")

def generate_answer(query):
    context = " ".join(search_similar(query))
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
    return qa_pipeline(prompt, max_new_tokens=100)[0]['generated_text']

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Device set to use cpu


In [25]:
generate_answer("What are the properties of life?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'Context: To make the whole-Earth\nimage, NASA scientists combine observations of different parts of the planet. (credit: modification of work by NASA)\nCHAPTER OUTLINE\nthey do not meet the criteria that biologists use to define life.\nFrom its earliest beginnings, biology has wrestled with four questions: What are the shared\nproperties that make something “alive”? How do those various living things function? When faced\nwith the remarkable diversity of life, how do we organize the different kinds of organisms so that\nwe can better understand them? And, finally—what biologists ultimately seek to understand—how\ndid this diversity arise and how is it continuing? As new organisms are discovered every day,\nbiologists continue to seek answers to these and other questions.\nProperties of Life\nAll groups of living organisms share several key characteristics or functions: order, sensitivity or\nresponse to stimuli, reproduction, adaptation, growth and development, regulation/homeostasis,

In [26]:
generate_answer("How do cells maintain homeostasis?")

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


"Context: These conditions may, however, change from one moment to the next. Organisms are able to\nmaintain homeostatic internal conditions within a narrow range almost constantly, despite environmental changes,\nby activation of regulatory mechanisms. These genes provide\ninstructions that will direct cellular growth and development, ensuring that a species’ young (Figure 1.4) will grow\nup to exhibit many of the same characteristics as its parents.\n1.1 • Themes and Concepts of Biology\n7\nFIGURE 1.4 Although no two look alike, these kittens have inherited genes from both parents and share many of the same characteristics.\n(credit: Pieter & Renée Lanser)\nRegulation/Homeostasis\nEven the smallest organisms are complex and require multiple regulatory mechanisms to coordinate internal\nfunctions, such as the transport of nutrients, response to stimuli, and coping with environmental stresses.\nHomeostasis (literally, “steady state”) refers to the relatively stable internal environment